# Combined RQ1 — Classification performance

**RQ1:** How accurately do Qwen, InternVL, and LLaVA classify safe, potentially unsafe, and unsafe situations across HuRoN, PAL, and CrowdBot?

Sequence: **Descriptive results → pooled per-class Precision/Recall/F1 → RQ1a approach comparison → RQ1b model comparison → RQ1c global comparison**. Macro-F1 is primary; Accuracy is secondary. Pooled frame metrics are descriptive, while inferential tests use the same 25 bags as paired blocks.

## 1. Setup and reusable functions

In [ ]:
from pathlib import Path
from itertools import combinations
import warnings
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import friedmanchisquare, rankdata, wilcoxon
from sklearn.metrics import confusion_matrix

HERE = Path.cwd().resolve()
ROOT = next((
    p for p in [HERE, *HERE.parents]
    if (p / "case_studies").is_dir() and (p / "analysis").is_dir()
), None)

if ROOT is None:
    raise RuntimeError("Could not locate repository root.")

CLEAN = ROOT / "analysis" / "combined" / "clean"
OUT = ROOT / "results" / "rq1"
TEX = OUT / "latex"
OUT.mkdir(parents=True, exist_ok=True)
TEX.mkdir(parents=True, exist_ok=True)
MODELS=['qwen','internvl','llava']; APPROACHES=['appfr','appod']; HISTORIES=list(range(2,33,2))
LABELS=['safe','potentially_unsafe','unsafe']; METRICS=['accuracy','macro_f1']
MN={'qwen':'Qwen','internvl':'InternVL','llava':'LLaVA'}; AN={'appfr':'AppFr','appod':'AppOd'}
HN=[f'H{h:02d}' for h in HISTORIES]; CN={'safe':'S','potentially_unsafe':'P','unsafe':'U'}

def export(df,name,caption,index=False,escape=True):
    df.to_csv(OUT/f'{name}.csv',index=index)
    (TEX/f'{name}.tex').write_text(df.to_latex(index=index,escape=escape,na_rep='--',float_format=lambda x:f'{x:.4f}',caption=caption,label=f'tab:{name}',position='tbp',multicolumn=True,multicolumn_format='c'),encoding='utf-8')

def score(y,p):
    cm=confusion_matrix(y,p,labels=LABELS); n=cm.sum(); rows=[]
    for i,c in enumerate(LABELS):
        tp=cm[i,i]; fn=cm[i].sum()-tp; fp=cm[:,i].sum()-tp; tn=n-tp-fn-fp
        pr=tp/(tp+fp) if tp+fp else 0.; rc=tp/(tp+fn) if tp+fn else 0.; f1=2*pr*rc/(pr+rc) if pr+rc else 0.
        rows.append({'class_label':c,'support':int(tp+fn),'precision':pr,'recall':rc,'f1':f1,'class_accuracy':(tp+tn)/n})
    c=pd.DataFrame(rows); return {'frames':int(n),'accuracy':np.trace(cm)/n,'macro_precision':c.precision.mean(),'macro_recall':c.recall.mean(),'macro_f1':c.f1.mean(),'balanced_accuracy':c.recall.mean()},c,cm

def holm(p):
    p=np.asarray(p,float); o=np.argsort(p); a=np.empty(len(p)); run=0.
    for k,i in enumerate(o): run=max(run,(len(p)-k)*p[i]); a[i]=min(run,1.)
    return a

def rrb(x,y):
    d=np.asarray(x,float)-np.asarray(y,float); d=d[d!=0]
    if not len(d): return 0.
    r=rankdata(abs(d)); return float((r[d>0].sum()-r[d<0].sum())/r.sum())

def pw(x,y):
    if np.allclose(x,y): return 0.,1.
    with warnings.catch_warnings(): warnings.simplefilter('ignore'); z=wilcoxon(x,y,zero_method='wilcox',alternative='two-sided',method='auto')
    return float(z.statistic),float(z.pvalue)

print('Input:',CLEAN); print('CSV:',OUT); print('LaTeX:',TEX)


## 2. Load, validate, and calculate metrics

In [ ]:
O=[]; C=[]; CM=[]; B=[]; D=[]
for m in MODELS:
 for a in APPROACHES:
  for h in HISTORIES:
   f=CLEAN/m/f'{m}_{a}_h{h:02d}.csv'; assert f.is_file(),f'Missing {f}'
   d=pd.read_csv(f,low_memory=False); assert len(d)==34912 and d.global_bag_id.nunique()==25
   assert not d.duplicated(['global_bag_id','frame_id']).any()
   o,c,cm=score(d.ground_truth,d.predicted_label); base={'model':m,'approach':a,'history':f'H{h:02d}','history_frames':h}
   O.append({**base,**o}); C += [{**base,**r} for r in c.to_dict('records')]
   CM += [{**base,'true_label':t,'predicted_label':p,'count':int(cm[i,j])} for i,t in enumerate(LABELS) for j,p in enumerate(LABELS)]
   for (case,bag),g in d.groupby(['case_study','global_bag_id']): q,_,_=score(g.ground_truth,g.predicted_label); B.append({**base,'case_study':case,'global_bag_id':bag,**q})
   for case,g in d.groupby('case_study'): q,_,_=score(g.ground_truth,g.predicted_label); D.append({**base,'case_study':case,**q})
overall=pd.DataFrame(O); per_class=pd.DataFrame(C); confusions=pd.DataFrame(CM); bag_metrics=pd.DataFrame(B); case_metrics=pd.DataFrame(D)
assert len(overall)==96 and len(bag_metrics)==2400 and len(case_metrics)==288
print('Validated: 96 files, 34,912 frames and 25 bags per configuration.')


## 3. Descriptive performance

The pooled table weights every frame equally. The bag summary gives each bag equal weight and reports medians and interquartile ranges.

In [ ]:
main=overall.pivot(index='history',columns=['model','approach'],values=['accuracy','macro_f1']).reindex(HN)
main.columns=pd.MultiIndex.from_tuples([(MN[m],AN[a],{'accuracy':'Accuracy','macro_f1':'Macro-F1'}[v]) for v,m,a in main.columns],names=['Model','Approach','Metric'])
main_pct=(100*main).round(2); main_pct.index.name='History'
bag_summary=bag_metrics.groupby(['model','approach','history'],as_index=False).agg(bags=('global_bag_id','nunique'),median_accuracy=('accuracy','median'),q1_accuracy=('accuracy',lambda x:x.quantile(.25)),q3_accuracy=('accuracy',lambda x:x.quantile(.75)),median_macro_f1=('macro_f1','median'),q1_macro_f1=('macro_f1',lambda x:x.quantile(.25)),q3_macro_f1=('macro_f1',lambda x:x.quantile(.75)))
display(main_pct); display(bag_summary.head())
export(overall,'rq1_descriptive_pooled','Pooled combined RQ1 performance.')
export(main_pct,'rq1_descriptive_main_percent','Pooled Accuracy and Macro-F1 in percent by model, approach, and history.',index=True)
export(bag_summary,'rq1_descriptive_bag_summary','Bag-level median and interquartile performance.')
export(case_metrics,'rq1_descriptive_case_study','Performance stratified by case study.')
export(bag_metrics,'rq1_bag_metrics','Bag-level Accuracy and Macro-F1 used in paired tests.')


## 4. Pooled per-class Precision, Recall, and F1

Each table contains 16 histories × 18 model–approach–class columns and reports percentages.

In [ ]:
export(per_class,'rq1_per_class_detailed','Detailed pooled per-class RQ1 metrics.')
export(confusions,'rq1_confusion_matrices','Combined confusion matrices.')
class_tables={}
for metric,title in [('precision','Precision'),('recall','Recall'),('f1','F1-score')]:
 t=per_class.pivot(index='history',columns=['model','approach','class_label'],values=metric).reindex(index=HN,columns=pd.MultiIndex.from_product([MODELS,APPROACHES,LABELS]))*100
 t.columns=pd.MultiIndex.from_tuples([(MN[m],AN[a],CN[c]) for m,a,c in t.columns],names=['Model','Approach','Class']); t.index.name='History'; t=t.round(2); class_tables[metric]=t
 display(t); export(t,f'rq1_per_class_{metric}_percent',f'Pooled per-class {title} in percent across all three case studies.',index=True)


## 5. RQ1a — Within-model approach comparison

AppFr and AppOd are paired on the same 25 bags. Holm correction covers the 16 histories within each model–metric family. Positive rank-biserial correlation favours AppFr; negative values favour AppOd.

In [ ]:
R=[]
for m in MODELS:
 for h in HISTORIES:
  p=bag_metrics.query('model==@m and history_frames==@h').pivot(index='global_bag_id',columns='approach',values=METRICS)
  for metric in METRICS:
   x,y=p[(metric,'appfr')],p[(metric,'appod')]; w,pv=pw(x,y); e=rrb(x,y); md=float(np.median(x-y))
   R.append({'model':m,'history':f'H{h:02d}','history_frames':h,'metric':metric,'n_pairs':25,'wilcoxon_statistic':w,'raw_p':pv,'median_difference_appfr_minus_appod':md,'rank_biserial_appfr_minus_appod':e})
rq1a=pd.DataFrame(R); rq1a['holm_p']=np.nan
for _,i in rq1a.groupby(['model','metric']).groups.items(): rq1a.loc[i,'holm_p']=holm(rq1a.loc[i,'raw_p'])
rq1a['significant']=rq1a.holm_p<.05; rq1a['significant_better_approach']=np.where(~rq1a.significant,'No difference',np.where(rq1a.rank_biserial_appfr_minus_appod>0,'AppFr',np.where(rq1a.rank_biserial_appfr_minus_appod<0,'AppOd','No difference')))
stats=rq1a.pivot(index='history',columns=['model','metric'],values=['holm_p','rank_biserial_appfr_minus_appod']).reindex(HN)
winners=rq1a.pivot(index='history',columns=['model','metric'],values='significant_better_approach').reindex(index=HN,columns=pd.MultiIndex.from_product([MODELS,METRICS])); winners.columns=pd.MultiIndex.from_tuples([(MN[m],{'accuracy':'Accuracy','macro_f1':'Macro-F1'}[x]) for m,x in winners.columns]); winners.index.name='History'
display(rq1a); display(winners)
export(rq1a,'rq1a_appfr_vs_appod_wilcoxon','RQ1a paired AppFr--AppOd tests.')
export(stats,'rq1a_statistics_wide','RQ1a Holm-adjusted p-values and rank-biserial effects.',index=True)
export(winners,'rq1a_significant_better_approach','Significantly better approach; no difference is shown otherwise.',index=True)


## 6. RQ1b — Within-approach model comparison

There are 64 Friedman tests. Holm correction covers 16 histories within each approach–metric family. Only Holm-significant omnibus tests receive three pairwise Wilcoxon tests, with Holm correction across those three pairs.

In [ ]:
F=[]
for a in APPROACHES:
 for h in HISTORIES:
  for metric in METRICS:
   p=bag_metrics.query('approach==@a and history_frames==@h').pivot(index='global_bag_id',columns='model',values=metric)[MODELS]
   z=friedmanchisquare(*[p[m] for m in MODELS]); F.append({'approach':a,'history':f'H{h:02d}','history_frames':h,'metric':metric,'n_blocks':25,'friedman_chi2':float(z.statistic),'raw_p':float(z.pvalue),'kendalls_w':float(z.statistic/(25*2))})
rq1b_friedman=pd.DataFrame(F); rq1b_friedman['holm_p']=np.nan
for _,i in rq1b_friedman.groupby(['approach','metric']).groups.items(): rq1b_friedman.loc[i,'holm_p']=holm(rq1b_friedman.loc[i,'raw_p'])
rq1b_friedman['significant']=rq1b_friedman.holm_p<.05

P=[]
for o in rq1b_friedman.query('significant').itertuples():
 p=bag_metrics[(bag_metrics.approach==o.approach)&(bag_metrics.history_frames==o.history_frames)].pivot(index='global_bag_id',columns='model',values=o.metric)[MODELS]; local=[]
 for m1,m2 in combinations(MODELS,2):
  w,pv=pw(p[m1],p[m2]); e=rrb(p[m1],p[m2]); md=float(np.median(p[m1]-p[m2])); local.append({'approach':o.approach,'history':o.history,'history_frames':o.history_frames,'metric':o.metric,'friedman_holm_p':o.holm_p,'model_1':m1,'model_2':m2,'n_pairs':25,'wilcoxon_statistic':w,'raw_p':pv,'median_difference_model1_minus_model2':md,'rank_biserial_model1_minus_model2':e})
 for r,pa in zip(local,holm([r['raw_p'] for r in local])): r['holm_p']=pa; r['significant']=pa<.05; r['significant_better_model']='No difference' if pa>=.05 or r['rank_biserial_model1_minus_model2']==0 else (r['model_1'] if r['rank_biserial_model1_minus_model2']>0 else r['model_2'])
 P+=local
rq1b_posthoc=pd.DataFrame(P); assert len(rq1b_posthoc)==3*int(rq1b_friedman.significant.sum())

W=[]
for o in rq1b_friedman.itertuples():
 win='—'
 if o.significant:
  p=bag_metrics[(bag_metrics.approach==o.approach)&(bag_metrics.history_frames==o.history_frames)].groupby('model')[o.metric].median(); top=p[p==p.max()].index.tolist()
  if len(top)==1:
   q=rq1b_posthoc[(rq1b_posthoc.approach==o.approach)&(rq1b_posthoc.history==o.history)&(rq1b_posthoc.metric==o.metric)&((rq1b_posthoc.model_1==top[0])|(rq1b_posthoc.model_2==top[0]))]
   if len(q)==2 and q.significant_better_model.eq(top[0]).all(): win={'qwen':'Q','internvl':'I','llava':'L'}[top[0]]
 W.append({'approach':AN[o.approach],'metric':{'accuracy':'Accuracy','macro_f1':'Macro-F1'}[o.metric],'history':o.history,'winner':win})
winner_matrix=pd.DataFrame(W).pivot(index=['approach','metric'],columns='history',values='winner').reindex(columns=HN)
display(rq1b_friedman); display(rq1b_posthoc); display(winner_matrix)
export(rq1b_friedman,'rq1b_model_friedman','RQ1b Friedman model comparisons.')
export(rq1b_posthoc,'rq1b_model_pairwise_wilcoxon','RQ1b pairwise Wilcoxon tests after significant omnibus tests.')
export(winner_matrix,'rq1b_significant_model_winners','Significant model winners: Q=Qwen, I=InternVL, L=LLaVA, dash=no unique winner.',index=True)
print('Friedman:',len(rq1b_friedman),'Significant:',int(rq1b_friedman.significant.sum()),'Wilcoxon:',len(rq1b_posthoc))


## 7. RQ1c — Global model comparison

Models are aligned on approach × history × bag (800 matched values per model). This reproduces the established global analysis; interpret it alongside RQ1b because bags recur across configurations.

In [ ]:
GF=[]; GP=[]; GA=[]
for metric in METRICS:
 p=bag_metrics.pivot(index=['case_study','global_bag_id','approach','history'],columns='model',values=metric)[MODELS].dropna(); assert len(p)==800
 z=friedmanchisquare(*[p[m] for m in MODELS]); GF.append({'metric':metric,'matched_values':800,'friedman_chi2':float(z.statistic),'raw_p':float(z.pvalue),'kendalls_w':float(z.statistic/(800*2))})
 local=[]
 for m1,m2 in combinations(MODELS,2):
  w,pv=pw(p[m1],p[m2]); e=rrb(p[m1],p[m2]); md=float(np.median(p[m1]-p[m2])); local.append({'metric':metric,'model_1':m1,'model_2':m2,'n_pairs':800,'wilcoxon_statistic':w,'raw_p':pv,'median_difference_model1_minus_model2':md,'rank_biserial_model1_minus_model2':e})
 for r,pa in zip(local,holm([r['raw_p'] for r in local])): r['holm_p']=pa; r['significant']=pa<.05; r['significant_better_model']='No difference' if pa>=.05 or r['rank_biserial_model1_minus_model2']==0 else (r['model_1'] if r['rank_biserial_model1_minus_model2']>0 else r['model_2'])
 GP+=local; q=p.reset_index(); q.insert(0,'metric',metric); GA.append(q)
rq1c_friedman=pd.DataFrame(GF); rq1c_posthoc=pd.DataFrame(GP); rq1c_aligned=pd.concat(GA,ignore_index=True)
display(rq1c_friedman); display(rq1c_posthoc)
export(rq1c_friedman,'rq1c_global_model_friedman','RQ1c global Friedman model comparison.')
export(rq1c_posthoc,'rq1c_global_model_pairwise_wilcoxon','RQ1c global pairwise Wilcoxon comparisons.')
export(rq1c_aligned,'rq1c_global_aligned_values','RQ1c matched bag-level values used in the global analysis.')


## 8. Documents saved for RQ1

In [ ]:
S=[]
for case in sorted(bag_metrics.case_study.unique()):
 for metric in METRICS:
  p=bag_metrics.query('case_study==@case').pivot(index=['global_bag_id','approach','history'],columns='model',values=metric)[MODELS]
  z=friedmanchisquare(*[p[m] for m in MODELS]); S.append({'case_study':case,'metric':metric,'matched_values':len(p),'friedman_chi2':float(z.statistic),'p_value':float(z.pvalue),'kendalls_w':float(z.statistic/(len(p)*2))})
sensitivity=pd.DataFrame(S); display(sensitivity); export(sensitivity,'rq1_case_study_sensitivity','RQ1 model-comparison sensitivity by case study.')
csv={p.stem for p in OUT.glob('*.csv') if p.stem != 'rq1_output_manifest'}; tex={p.stem for p in TEX.glob('*.tex')}; assert csv==tex,(sorted(csv-tex),sorted(tex-csv))
manifest=pd.DataFrame({'table':sorted(csv),'csv':[str(OUT/f'{x}.csv') for x in sorted(csv)],'latex':[str(TEX/f'{x}.tex') for x in sorted(csv)]})
manifest.to_csv(OUT/'rq1_output_manifest.csv',index=False); display(manifest)
print(f'RQ1 COMPLETE: {len(csv)} result tables saved in both CSV and LaTeX formats.')
